# Fig 5 supp_9 (S5i) - within-family LOPO classifier (common and genewise precision)

Leave-one-PAIR-out: DRIMSeq feature-selection inside each fold (`~ family + Diagnosis`, top-50, no leakage) + ridge on isoform proportions, predicting the held-out pair's 2 samples.

Built for BOTH precision modes.

**Common precision ran on all 9 folds; genewise crashed on 2 folds (non-finite optim on the small family-blocked design)** - so common is the more robust choice here. n=9 pairs (18 samples), proof-of-concept: the pooled-AUC CI crosses chance; the within-pair ordering (proband > sibling) is the better-powered read.

Inputs: common = rebuilt bug-fixed `LOPO_siblings_common/` (`prec_moderation="none"` fix -> all 9 folds; 9/9, AUC 0.79); genewise = archived `within_family_systematic/LOPO_genewise/`.


## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment" / "downstr.yml").is_file())
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")

In [ ]:
import sys, warnings

warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

sys.path.insert(0, f"{REPO_ROOT}/figures/style")
import paper_fig_style as S

S.apply_rcparams()
BASE = f"{DATA_ROOT}/3utr/Blood/2025_analyses/within_family_systematic"
COMMON_BASE = f"{DATA_ROOT}/3utr/Blood/2025_analyses/LOPO_siblings_common"  # rebuilt bug-fixed common-precision LOPO (prec_moderation="none"); genewise stays archived
ASD, CTRL, GRN, RED = "#d73027", "#4575b4", "#1a9850", "#d7191c"


def load(mode):
    pr = pd.read_csv(
        f"{COMMON_BASE}/lopo_preds.csv"
        if mode == "common"
        else f"{BASE}/LOPO_{mode}/lopo_preds.csv"
    )
    y = (pr.truth == "ASD").astype(int).values
    fpr, tpr, _ = roc_curve(y, pr.prob_ASD.values)
    A = auc(fpr, tpr)
    rng = np.random.default_rng(0)
    bs = []
    for _ in range(2000):
        idx = rng.integers(0, len(pr), len(pr))
        if len(np.unique(y[idx])) < 2:
            continue
        f2, t2, _ = roc_curve(y[idx], pr.prob_ASD.values[idx])
        bs.append(auc(f2, t2))
    lo, hi = np.percentile(bs, [2.5, 97.5])
    acc = np.mean(
        (pr.prob_ASD >= 0.5).map({True: "ASD", False: "CTRL"}).values == pr.truth.values
    )
    piv = pr.pivot_table(index="family", columns="truth", values="prob_ASD")
    piv["correct"] = piv["ASD"] > piv["CTRL"]
    piv = piv.sort_values("ASD")
    return dict(
        pr=pr,
        fpr=fpr,
        tpr=tpr,
        A=A,
        lo=lo,
        hi=hi,
        acc=acc,
        piv=piv,
        npair=int(piv.correct.sum()),
        nfold=len(piv),
    )


def build_panel(mode):
    R = load(mode)
    PANEL = f"fig_s5i_lopo_within_family_{mode}"
    fig, ax = plt.subplots(
        1,
        2,
        figsize=(S.FULL_W_PX / 300, S.mm_to_px(55) / 300),
        gridspec_kw={"width_ratios": [1, 1.15], "wspace": 0.34},
    )
    ax[0].plot(R["fpr"], R["tpr"], color="#2c7fb8", lw=1.8)
    ax[0].fill_between(R["fpr"], R["tpr"], 0, color="#2c7fb8", alpha=0.10)
    ax[0].plot([0, 1], [0, 1], ls="--", c="grey", lw=0.7)
    ax[0].text(
        0.40,
        0.10,
        f"pooled AUC = {R['A']:.2f}\n95% CI {R['lo']:.2f}-{R['hi']:.2f}\n(n={len(R['pr'])}; CI crosses 0.5)",
        fontsize=S.FONT_ANNOT,
    )
    ax[0].set_xlabel("False positive rate", fontsize=S.FONT_LABEL)
    ax[0].set_ylabel("True positive rate", fontsize=S.FONT_LABEL)
    ax[0].set_title("(a) leave-one-pair-out ROC", fontsize=S.FONT_ANNOT)
    ax[0].set_xlim(-0.02, 1.02)
    ax[0].set_ylim(-0.02, 1.02)
    yy = np.arange(R["nfold"])
    for i, (fam, r) in enumerate(R["piv"].iterrows()):
        col = GRN if r["correct"] else RED
        ax[1].plot([r["CTRL"], r["ASD"]], [i, i], color=col, lw=1.0, zorder=1)
        ax[1].scatter(
            r["CTRL"], i, s=22, c=CTRL, edgecolors="k", linewidths=0.3, zorder=2
        )
        ax[1].scatter(
            r["ASD"], i, s=22, c=ASD, edgecolors="k", linewidths=0.3, zorder=2
        )
    ax[1].axvline(0.5, ls=":", c="grey", lw=0.7)
    ax[1].set_yticks(yy)
    ax[1].set_yticklabels(
        [f"P{i+1}" for i in range(R["nfold"])], fontsize=S.FONT_TICK - 1
    )
    ax[1].set_xlabel("predicted prob(ASD), out-of-fold", fontsize=S.FONT_LABEL)
    ax[1].set_ylabel("held-out family", fontsize=S.FONT_LABEL)
    ax[1].set_title(
        f"(b) within-pair ({R['npair']}/{R['nfold']} correct)", fontsize=S.FONT_ANNOT
    )
    ax[1].scatter(
        [], [], s=22, c=ASD, edgecolors="k", linewidths=0.3, label="ASD proband"
    )
    ax[1].scatter(
        [], [], s=22, c=CTRL, edgecolors="k", linewidths=0.3, label="unaffected sibling"
    )
    ax[1].legend(fontsize=S.FONT_LEGEND, frameon=False, loc="lower right")
    for a in ax:
        a.tick_params(labelsize=S.FONT_TICK)
        [s.set_visible(False) for s in (a.spines["top"], a.spines["right"])]
    fig.suptitle(
        f"Within-family LOPO classifier ({mode} precision; DRIMSeq ~family+Diagnosis feature-sel inside fold, top-50, ridge) - n=9 pairs, proof-of-concept",
        fontsize=S.FONT_ANNOT - 0.5,
    )
    fig.subplots_adjust(left=0.07, right=0.985, bottom=0.16, top=0.86)
    S.save_and_show(fig, PANEL, figure="Fig5/supp_9")
    return R

In [ ]:
Rc = build_panel("common")

In [ ]:
Rg = build_panel("genewise")

In [ ]:
# (c) common vs genewise comparison
import os


def feats(mode):
    p = (
        f"{COMMON_BASE}/lopo_feat_sel.csv"
        if mode == "common"
        else f"{BASE}/LOPO_{mode}/lopo_feat_sel.csv"
    )
    return set(pd.read_csv(p)["feature_id"]) if os.path.exists(p) else set()


fc, fg = feats("common"), feats("genewise")
jac = len(fc & fg) / max(len(fc | fg), 1)
fig, ax = plt.subplots(figsize=(S.HALF_W_PX / 300, S.mm_to_px(52) / 300))
labs = ["common", "genewise"]
A = [Rc["A"], Rg["A"]]
lo = [Rc["lo"], Rg["lo"]]
hi = [Rc["hi"], Rg["hi"]]
x = np.arange(2)
ax.bar(
    x,
    A,
    color=["#2c7fb8", "#80b1d3"],
    width=0.6,
    yerr=[np.array(A) - np.array(lo), np.array(hi) - np.array(A)],
    capsize=3,
    error_kw=dict(lw=0.7),
)
ax.axhline(0.5, ls="--", c="grey", lw=0.7)
for i, a in enumerate(A):
    ax.text(i, hi[i] + 0.02, f"{a:.2f}", ha="center", fontsize=S.FONT_TICK)
ax.set_xticks(x)
ax.set_xticklabels(labs, fontsize=S.FONT_TICK)
ax.set_ylim(0, 1.05)
ax.set_ylabel("pooled out-of-fold AUC", fontsize=S.FONT_LABEL)
ax.set_title("(c) common vs genewise precision", fontsize=S.FONT_ANNOT)
txt = (
    f"folds OK: common {Rc['nfold']}/9, genewise {Rg['nfold']}/9 (2 optim failures)\n"
    f"within-pair: common {Rc['npair']}/{Rc['nfold']}, genewise {Rg['npair']}/{Rg['nfold']}\n"
    f"selected-feature Jaccard = {jac:.2f}\n-> common precision is MORE ROBUST here (no fold crashes) and comparable"
)
ax.text(
    0.5,
    -0.34,
    txt,
    transform=ax.transAxes,
    ha="center",
    va="top",
    fontsize=S.FONT_ANNOT - 1,
)
ax.tick_params(labelsize=S.FONT_TICK)
[s.set_visible(False) for s in (ax.spines["top"], ax.spines["right"])]
fig.subplots_adjust(left=0.16, right=0.96, bottom=0.42, top=0.88)
S.save_and_show(fig, "fig_s5i_lopo_common_vs_genewise", figure="Fig5/supp_9")
print(
    f"common AUC={Rc['A']:.3f} within-pair {Rc['npair']}/{Rc['nfold']} | genewise AUC={Rg['A']:.3f} within-pair {Rg['npair']}/{Rg['nfold']} | feat Jaccard={jac:.2f}"
)